# LendItEazy Power BI Dashboard Plan

## Scope
This document translates the five tables below into a complete Power BI reporting architecture:

1. customer_profile.csv
2. loan_details.csv
3. loan_outcome.csv
4. repayment_behavior.csv
5. risk_scored_portfolio.csv

The design is based on the actual table structure and the observed business patterns in the portfolio:
- Gig workers are higher risk than salaried customers.
- BNPL is the riskiest product.
- Social Media is riskier than Referral.
- Tier 3 cities are underpriced relative to risk.
- Early warning signals appear in months 1 to 3.

---

## 1. Data Model

### Fact and dimension roles
| Table | Grain | Role |
|---|---|---|
| customer_profile | 1 row per customer | Customer dimension |
| loan_details | 1 row per loan | Origination fact |
| loan_outcome | 1 row per loan | Credit outcome fact |
| repayment_behavior | 1 row per loan-month | Collections fact |
| risk_scored_portfolio | 1 row per loan | Scorecard / analytics fact |

### Required relationships
| From | To | Cardinality | Notes |
|---|---|---|---|
| customer_profile[customer_id] | loan_details[customer_id] | 1 to many | Main customer-to-loan link |
| loan_details[loan_id] | loan_outcome[loan_id] | 1 to 1 | One outcome per loan |
| loan_details[loan_id] | repayment_behavior[loan_id] | 1 to many | Monthly repayment history |
| customer_profile[customer_id] | risk_scored_portfolio[customer_id] | 1 to many | Customer analytics by risk |
| loan_details[loan_id] | risk_scored_portfolio[loan_id] | 1 to 1 | Scorecard grain |
| Date table | all date fields | 1 to many | Use inactive relationships for secondary dates |

### Recommended date table
Use a dedicated Date table with relationships to:
- onboarding_date
- origination_date
- disbursement_date
- due_date
- payment_date
- closure_date

### Core calculated columns
- Age Band
- Income Band
- Credit Score Band
- EMI to Income Band
- DPD Bucket
- Risk Score Band
- Risk Tier Label
- Decision Label
- Acquisition Cohort
- Default Vintage

---

## 2. Recommended Dashboard Suite

### Dashboard pages
| Page | Purpose | Primary audience |
|---|---|---|
| Executive Overview | Portfolio health and board summary | CEO, CFO, CRO |
| Customer Analytics and Acquisition | Borrower profile and source quality | Marketing, growth, CX |
| Risk and Underwriting | Scorecard, approval quality, policy control | Credit policy, underwriting |
| Collections and Early Warning | DPD, bounce, delinquency, recovery | Collections, operations |
| Pricing and Profitability | Risk-based pricing and margin quality | Finance, product, risk |
| Forecasting and Alerts | Trend, anomaly, and forward look | Leadership, risk ops |

### Not supported by the dataset
| Dashboard type | Support | Reason |
|---|---|---|
| HR Dashboard | No | No employee or workforce table |
| Inventory Dashboard | No | No stock, warehouse, or product inventory table |

---

## 3. Dashboard 1 - Executive Overview

### Title
LendItEazy Executive Portfolio Overview

### Business objective
Provide leadership with a single-page summary of portfolio size, growth, risk, defaults, and recovery.

### Target users
CEO, CFO, CRO, board reporting team

### Key insights
- Portfolio growth by month
- Default concentration by product, channel, and customer type
- Recovery effectiveness
- Risk mix across the book

### KPI cards
| KPI Name | Formula | Business meaning | Target benchmark |
|---|---|---|---|
| Total Customers | DISTINCTCOUNT(customer_profile[customer_id]) | Active borrower base | Growth over prior month |
| Total Loans | DISTINCTCOUNT(loan_details[loan_id]) | Portfolio size | Growth over prior period |
| Total Disbursed Amount | SUM(loan_details[loan_amount]) | Gross book size | Increase with controlled risk |
| Portfolio Default Rate | Defaults / Total loans | Core portfolio health metric | Below the current 20.5% baseline |
| Recovery Rate | Recovery amount / defaulted exposure | Collections efficiency | Improve quarterly |

### Recommended slicers
| Field | Select mode | Control | Why useful |
|---|---|---|---|
| origination_date | Range | Between slider | Controls time period |
| product_type | Multi select | Dropdown | Compare products |
| city_tier | Multi select | Dropdown | Compare risk geography |
| employment_type | Multi select | Dropdown | Compare customer risk |
| acquisition_channel | Multi select | Dropdown | Compare source quality |

### Charts and visuals
| Chart type | X-axis | Y-axis | Legend | Tooltip fields | Purpose | Why suitable |
|---|---|---|---|---|---|---|
| Line and clustered column chart | Month | Loan count and disbursed amount | Product type | Default rate, avg risk score, avg interest rate | Show growth and mix in one view | Best for executive trend reading |
| Clustered bar chart | Employment type | Default rate | City tier | Avg credit score, avg income | Show segment risk | Best for category comparison |
| 100% stacked column chart | Acquisition channel | Share of loans | Risk tier | Default rate, avg risk score | Show source mix | Best for composition analysis |
| Heatmap matrix | City tier | Product type | Conditional formatting | Loan count, default rate, avg EMI | Show risk concentration | Best for 2D comparison |

### Layout suggestion
- Top row: KPI cards
- Middle left: trend chart
- Middle right: channel and risk mix
- Bottom: heatmap and segment bar chart
- Right side: short executive narrative

### Drill-through pages
- Customer detail
- Loan detail
- Default detail
- Collections detail

### Conditional formatting
- Red for high default rates
- Amber for borderline risk
- Green for low-risk segments

### Mobile layout
- KPI strip first
- One trend chart
- One key risk bar
- Avoid dense matrices on mobile

### AI / forecasting ideas
- Smart Narrative
- Forecast monthly disbursement and default trend
- Anomaly detection on default spikes

---

## 4. Dashboard 2 - Customer Analytics and Acquisition

### Title
Customer Profile and Acquisition Analytics

### Business objective
Understand borrower mix, source quality, and which acquisition paths attract stronger or weaker customers.

### Target users
Marketing, growth, CRM, business heads

### Key insights
- Borrower mix by age, income, education, and city tier
- Channel quality by default rate
- Customer quality by acquisition source

### KPI cards
| KPI Name | Formula | Business meaning | Target benchmark |
|---|---|---|---|
| New Customers | DISTINCTCOUNT(customer_profile[customer_id]) by onboarding_date | Acquisition volume | Month-over-month growth |
| Avg Monthly Income | AVERAGE(customer_profile[monthly_income]) | Borrower affordability | Higher is better |
| Avg Credit Score | AVERAGE(customer_profile[credit_score]) | Borrower quality | Higher is better |
| Referral Share | Referral customers / total customers | Best-quality source share | Grow over time |
| Social Media Share | Social Media customers / total customers | Higher-risk acquisition share | Keep under control |

### Recommended slicers
| Field | Select mode | Control | Why useful |
|---|---|---|---|
| onboarding_date | Range | Between slider | Cohort view |
| city_tier | Multi select | Buttons | Compare geography |
| employment_type | Multi select | Dropdown | Compare borrower types |
| acquisition_channel | Multi select | Dropdown | Compare source quality |
| education_level | Multi select | Dropdown | Compare profile mix |

### Charts and visuals
| Chart type | X-axis | Y-axis | Legend | Tooltip fields | Purpose | Why suitable |
|---|---|---|---|---|---|---|
| Clustered column chart | Employment type | Customer count | City tier | Avg income, avg credit score | Show borrower mix | Best for segmentation |
| Scatter chart | Monthly income | Credit score | Employment type | Age, city tier, acquisition channel | Show profile clusters and outliers | Best for relationship analysis |
| Bar chart | Acquisition channel | Default rate | None or product type | Avg income, avg credit score | Show channel quality | Best for source comparison |
| Matrix heatmap | Education level | Employment type | None | Customer count, avg credit score | Show customer profile quality | Best for multi-dimensional profiling |

### Layout suggestion
- Left: demographic mix
- Center: scatter and channel analysis
- Right: profile heatmap and KPI strip

### Drill-through pages
- Customer 360
- Channel performance detail
- High-risk customer detail

### Alerts
- Social Media default rate too high
- Referral share dropping
- High-risk customer share rising

### AI / forecasting ideas
- Key Influencers on default_flag
- Q&A visual for channel quality questions
- Decomposition tree for customer quality drivers

---

## 5. Dashboard 3 - Risk and Underwriting

### Title
Credit Risk, Scorecard, and Underwriting Control Tower

### Business objective
Monitor approval quality, risk score calibration, and default concentration.

### Target users
Credit policy, underwriting, CRO, risk analytics

### Key insights
- Approval vs reject mix
- Risk tier distribution
- Default concentration by product, city, and channel
- Scorecard calibration quality

### KPI cards
| KPI Name | Formula | Business meaning | Target benchmark |
|---|---|---|---|
| Approval Rate | Approved loans / total loans | Underwriting strictness | Balance growth and risk |
| Reject Rate | Rejected loans / total loans | Policy strictness | Depends on growth strategy |
| High Risk Share | High-risk loans / total loans | Risk exposure | Trend downward |
| Avg Risk Score | AVERAGE(risk_scored_portfolio[risk_score]) | Risk pressure | Lower is better |
| Default Rate on Approved Loans | Defaults among APPROVE decisions / approved loans | Approval quality | Below portfolio average |

### Recommended slicers
| Field | Select mode | Control | Why useful |
|---|---|---|---|
| risk_tier | Multi select | Dropdown | Risk focus |
| risk_grade | Multi select | Dropdown | Policy comparison |
| product_type | Multi select | Dropdown | Product underwriting |
| city_tier | Multi select | Dropdown | Geography policy |
| decision | Multi select | Buttons or dropdown | Compare approve/review/reject |

### Charts and visuals
| Chart type | X-axis | Y-axis | Legend | Tooltip fields | Purpose | Why suitable |
|---|---|---|---|---|---|---|
| Scatter chart | EMI to income | risk_probability | risk_tier | default_flag, scorecard_points, credit_score | Show score-risk calibration | Best for underwriting analysis |
| Clustered bar chart | Product type | Default rate | City tier | Avg risk score, approval rate | Show product risk | Best for policy tuning |
| Heatmap matrix | City tier | Employment type | Conditional formatting | risk_score, default_rate, approval_rate | Show risk hotspots | Best for policy review |
| Decomposition tree | Default rate | Product, channel, city tier, employment | None | risk_score, approval rate | Root-cause analysis | Best for deep investigation |

### Layout suggestion
- Top: KPI strip
- Left: score-risk scatter
- Center: product risk bars
- Right: heatmap
- Bottom: decomposition tree

### Drill-through pages
- Scorecard detail
- High-risk borrower detail
- Rejected application detail

### Conditional formatting
- Red for high-risk tiers
- Amber for review
- Green for low-risk approvals
- Icon sets for decision categories

### AI / forecasting ideas
- Key Influencers on default_flag
- Smart Narrative for policy summary
- What-if parameters for pricing or cutoff changes

---

## 6. Dashboard 4 - Collections and Early Warning

### Title
Delinquency, Bounce, and Early Warning Management

### Business objective
Track repayment stress, bounce behavior, and early signs of default.

### Target users
Collections, operations, recovery, risk teams

### Key insights
- DPD progression
- Bounce frequency
- Early warning capture rate
- Default precursors in months 1 to 3

### KPI cards
| KPI Name | Formula | Business meaning | Target benchmark |
|---|---|---|---|
| Avg DPD First 3 Months | Average of avg_dpd_first_3_months | Early repayment stress | Lower is better |
| Bounce Rate | Bounced payments / total repayments | Payment friction | Keep low |
| Late Payment Count First 3 Months | Average late_payment_count_first_3_months | Early delinquency count | Lower is better |
| Early Warning Capture Rate | Defaulters with 2+ early signals / total defaulters | Warning effectiveness | Increase over time |
| Recovery Rate | Recovery amount / defaulted exposure | Collection success | Improve quarter over quarter |

### Recommended slicers
| Field | Select mode | Control | Why useful |
|---|---|---|---|
| month_number | Single or range | Slider | Track repayment age |
| product_type | Multi select | Dropdown | Compare products |
| employment_type | Multi select | Dropdown | Compare borrower groups |
| risk_tier | Multi select | Dropdown | Focus on stressed accounts |
| payment_status | Multi select | Dropdown | Filter delinquency stage |

### Charts and visuals
| Chart type | X-axis | Y-axis | Legend | Tooltip fields | Purpose | Why suitable |
|---|---|---|---|---|---|---|
| Line chart | Month number | Avg DPD | Default flag | Bounce rate, payment status mix | Show delinquency trend | Best for time series stress |
| Stacked column chart | Month number | Loan count | Payment status | Bounce count, amount_due | Show deterioration stages | Best for operational monitoring |
| Bar chart | Product type | Bounce rate | None or employment type | Avg DPD, recovery rate | Compare collections risk | Best for prioritization |
| Histogram or column chart | Stress score band | Loan count | Default flag | Avg DPD, bounce rate | Show early warning distribution | Best for threshold tuning |

### Layout suggestion
- Top: KPI strip
- Left: DPD trend
- Center: payment status stack
- Right: bounce comparison
- Bottom: stress score distribution

### Drill-through pages
- Loan repayment history
- Customer delinquency timeline
- Default case file

### Conditional formatting
- 1 to 30 DPD: amber
- 31 to 60 DPD: orange
- 61+ DPD: red
- Highlight repeated bounces with icons

### AI / forecasting ideas
- Forecast DPD trend
- Anomaly detection on bounce spikes
- Smart Narrative for collections commentary

---

## 7. Dashboard 5 - Pricing and Profitability

### Title
Risk-Based Pricing and Portfolio Profitability

### Business objective
Check whether interest rates, fees, and EMI burden are aligned with observed risk.

### Target users
Finance, product, pricing, risk

### Key insights
- Underpricing in Tier 3 cities
- Pricing gap by product and segment
- EMI burden versus risk
- Risk-adjusted yield by book segment

### KPI cards
| KPI Name | Formula | Business meaning | Target benchmark |
|---|---|---|---|
| Avg Interest Rate | AVERAGE(loan_details[interest_rate]) | Price level | Should reflect risk mix |
| Avg EMI to Income | AVERAGE(risk_scored_portfolio[emi_to_income]) | Affordability pressure | Lower is safer |
| Avg Processing Fee % | AVERAGE(processing_fee / loan_amount) | Upfront fee yield | Stable by product |
| Risk-Adjusted Yield | Interest yield minus expected loss | Net profitability | Positive and stable |
| Pricing Gap Index | Risk score vs interest rate gap | Detect underpriced segments | Close gap in Tier 3 and high risk |

### Recommended slicers
| Field | Select mode | Control | Why useful |
|---|---|---|---|
| product_type | Multi select | Dropdown | Pricing by product |
| risk_grade | Multi select | Dropdown | Risk-based pricing |
| city_tier | Multi select | Dropdown | Geography pricing |
| employment_type | Multi select | Dropdown | Segment pricing |
| decision | Multi select | Dropdown | Compare approved and rejected loans |

### Charts and visuals
| Chart type | X-axis | Y-axis | Legend | Tooltip fields | Purpose | Why suitable |
|---|---|---|---|---|---|---|
| Combo chart | Product type | Interest rate and default rate | Risk grade | Loan amount, EMI to income | Compare price and risk together | Best for pricing calibration |
| Scatter chart | EMI to income | Default rate or risk probability | City tier | Interest rate, loan amount | Show affordability-risk relationship | Best for price stress analysis |
| Clustered bar chart | Product type | Avg loan amount | Risk tier | Avg interest rate, default rate | Show value versus risk | Best for segment comparison |
| Line chart with forecast | Origination month | Default rate or yield | Product type | Risk score, disbursed amount | Show trend and forecast | Best for forward view |

### Layout suggestion
- Top: KPI strip
- Left: price versus risk combo
- Center: affordability scatter
- Right: product risk bars
- Bottom: forecast and pricing gap note

### Drill-through pages
- Pricing by product
- Pricing by city tier
- Underpriced borrower detail

### AI / forecasting ideas
- What-if pricing scenarios
- Forecast default rate by product
- Key Influencers for price-risk behavior

---

## 8. Best Page Structure

Recommended report flow:
1. Executive Overview
2. Customer Analytics and Acquisition
3. Risk and Underwriting
4. Collections and Early Warning
5. Pricing and Profitability
6. Forecasting and Alerts

Recommended page layout pattern:
- Top row: KPI cards
- Left: slicers
- Center: main visual
- Right: supporting visual or narrative
- Bottom: detail table or heatmap

---

## 9. Interactivity Recommendations

Best visuals to make interactive:
- Scatter charts
- Heatmaps and matrices
- Line charts
- Bar charts
- Decomposition trees
- Drill-through customer and loan pages

Lower interactivity priority:
- KPI cards
- Narrative text
- Static summary tables

---

## 10. Alerts to Configure

| Metric | Alert condition | Action |
|---|---|---|
| Portfolio Default Rate | Above 22% or rising sharply | Escalate to CRO |
| BNPL Default Rate | Above 30% | Review product policy |
| Social Media Default Rate | Above 25% | Tighten acquisition controls |
| Tier 3 Default Rate | Above 28% | Reprice or tighten underwriting |
| Bounce Rate | Above threshold | Trigger collections action |
| Avg DPD First 3 Months | Rising for 2 periods | Early warning escalation |
| High Risk Share | Increasing faster than approvals | Check policy leakage |
| Recovery Rate | Falling month over month | Review collections process |

---

## 11. Data Cleaning and Preprocessing

Recommended preparation steps:
- Standardize date columns and create one Date table.
- Normalize customer segments and city names.
- Convert flags to consistent 0/1 or Yes/No labels.
- Create Age Band, Income Band, Credit Band, DPD Bucket, Risk Tier Band.
- Ensure blank or zero income cases are handled consistently for students.
- Remove any duplicate keys before modeling.
- Hide technical IDs from business users.

---

## 12. Performance Optimization

- Use a star schema.
- Disable auto date/time.
- Hide unused columns.
- Prefer measures over repeated calculated columns when possible.
- Use incremental refresh for repayment_behavior if the table grows.
- Pre-aggregate common summary tables for executive pages.
- Avoid bi-directional filters unless necessary.

---

## 13. Executive Summary Section

The executive summary should answer three questions immediately:
1. Is the book growing or shrinking?
2. Which segments create the most risk?
3. Which actions are most urgent right now?

A concise narrative card should state:
- portfolio default rate
- riskiest product
- riskiest channel
- underpriced city tier
- early warning capture rate

---

## 14. Top 10 Business Questions

1. Which customer segments default the most?
2. Which acquisition channels bring the best borrowers?
3. Which products are underpriced relative to risk?
4. How strong are the early warning signals in months 1 to 3?
5. Which city tiers contribute the most defaults?
6. How does EMI burden affect default behavior?
7. Which approvals should have been rejected?
8. Which loans recover best after default?
9. Which channels look good on volume but poor on quality?
10. Where should underwriting and pricing be tightened first?

---

## 15. Top 10 Hidden Insights

1. Gig workers are materially riskier than salaried borrowers.
2. BNPL is the highest-risk product.
3. Social Media acquisition is significantly riskier than Referral.
4. Tier 3 customers appear underpriced relative to observed risk.
5. Months 1 to 3 carry strong early warning value.
6. High EMI to income loans cluster in riskier approvals.
7. Risk score and approval decision may not fully align in borderline cases.
8. Some channels may look good on volume but poor on quality.
9. Recovery behavior may differ by product and risk tier.
10. Risk is driven by interactions, not a single field alone.

---

## 16. Top 10 Advanced Power BI Features

1. Decomposition Tree for default driver analysis.
2. Key Influencers for default and approval drivers.
3. Smart Narrative for executive commentary.
4. Forecasting on monthly trend visuals.
5. Anomaly detection on delinquency spikes.
6. Drill-through pages for customer and loan detail.
7. Field parameters to switch between metrics.
8. What-if parameters for pricing and policy changes.
9. Row-level security if different teams need different access.
10. Incremental refresh for the repayment table.

---

## 17. Color Theme Recommendation

Best primary theme: light corporate theme.

Suggested palette:
- Navy: #0B1F3A
- Teal: #1F8A8A
- Gold: #C49A3A
- Amber: #E39D3C
- Red: #B33A3A
- Gray: #6B7280

Use red only for risk and alert states. Avoid overly bright or decorative colors.

---

## 18. Final Recommendation

If you are building this in Power BI, start with:
1. star schema modeling,
2. Executive Overview,
3. Risk and Underwriting,
4. Collections and Early Warning,
5. then Customer and Pricing pages.

That sequence gives management the fastest path to actionable business value.
